# Neural Networks with TensorFlow and NumPy

## Overview

This module introduces the practical implementation of neural networks using **TensorFlow** and **NumPy**. It covers how to represent input data, define neural network architectures, perform inference, and train models using TensorFlow's high-level API.

The module focuses on three main topics:

- Implementing inference using TensorFlow layers
- Understanding vectors, matrices, arrays, and tensors
- Building and training neural networks with the Sequential API

The goal is to understand both how to use TensorFlow efficiently and how its abstractions relate to the underlying computations.

---

## 1. Implementing Inference with TensorFlow

TensorFlow is a widely used framework for developing and training machine learning models. It provides layers and mathematical operations that simplify neural network implementation.

Rather than manually implementing every neuron's computation, we can define layers using TensorFlow's `Dense` class.

A dense layer connects each neuron to all outputs from the preceding layer.

### Example: Coffee Roasting Prediction

Imagine a model that predicts whether coffee beans will be well roasted based on two input features:

- Roasting temperature
- Roasting duration

The neural network contains two layers:

| Layer | Number of units | Activation |
|---|---:|---|
| Hidden layer | 3 | Sigmoid |
| Output layer | 1 | Sigmoid |

The input contains two numerical features. The first layer transforms them into three activations, and the second layer produces one output.

```python
import numpy as np
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        units=3,
        activation="sigmoid"
    ),
    tf.keras.layers.Dense(
        units=1,
        activation="sigmoid"
    )
])

# Example input: temperature and roasting duration
X = np.array([
    [200.0, 12.0]
])

prediction = model(X)

print(prediction.numpy())
```

This example illustrates the architecture and inference workflow. The model must be trained, or initialized with trained weights and biases, before its output can be used as a meaningful coffee-roasting prediction.

The output of the final sigmoid layer is a value between 0 and 1.

For a binary classification task, a threshold such as 0.5 can be used to assign a class:

```python
predicted_class = int(prediction.numpy()[0, 0] >= 0.5)
```

The threshold converts a probability-like score into a binary decision. It does not guarantee that the prediction is correct.

---

## 2. Handwritten Digit Classification

The same approach can be applied to image classification.

For example, consider a neural network that distinguishes handwritten zeros from ones using an image of size \(8 \times 8\) pixels.

The image is represented as 64 numerical features after flattening.

A possible architecture is:

| Layer | Units | Activation |
|---|---:|---|
| Hidden layer 1 | 25 | Sigmoid |
| Hidden layer 2 | 15 | Sigmoid |
| Output layer | 1 | Sigmoid |

The inference pipeline is:

```text
8 × 8 image
     |
     v
64 pixel features
     |
     v
Dense(25, sigmoid)
     |
     v
Dense(15, sigmoid)
     |
     v
Dense(1, sigmoid)
     |
     v
Probability of digit 1
```

Example implementation:

```python
import numpy as np
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64,)),
    tf.keras.layers.Dense(25, activation="sigmoid"),
    tf.keras.layers.Dense(15, activation="sigmoid"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Example input containing one flattened image.
image = np.zeros((1, 64), dtype=np.float32)

prediction = model.predict(image, verbose=0)

predicted_digit = int(prediction[0, 0] >= 0.5)

print("Probability of digit 1:", prediction[0, 0])
print("Predicted class:", predicted_digit)
```

The example demonstrates the prediction interface. Since the model has not been trained, its output is not a meaningful digit classification yet.

---

## 3. Understanding NumPy Arrays and TensorFlow Tensors

Correct data representation is essential when implementing neural networks.

### Matrices

A matrix is a two-dimensional structure organized into rows and columns.

For example, a matrix with two rows and three columns has shape:

\[
(2,3)
\]

In NumPy:

```python
import numpy as np

matrix = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print(matrix.shape)
# (2, 3)
```

The first dimension represents the number of rows, and the second represents the number of columns.

### Vectors and one-dimensional arrays

A vector can be represented mathematically as either a row vector or a column vector.

```python
row_vector = np.array([[1, 2, 3]])
column_vector = np.array([[1], [2], [3]])
one_dimensional = np.array([1, 2, 3])

print(row_vector.shape)       # (1, 3)
print(column_vector.shape)    # (3, 1)
print(one_dimensional.shape)  # (3,)
```

A NumPy array with shape `(3,)` is one-dimensional. Unlike `(1, 3)` or `(3, 1)`, it does not explicitly represent a row or column.

These shapes are not interchangeable in every mathematical operation. Understanding the distinction helps prevent shape-related errors.

### TensorFlow tensors

TensorFlow uses tensors to represent data of different ranks:

- Rank 0: scalar
- Rank 1: one-dimensional tensor
- Rank 2: matrix
- Rank 3 or higher: multidimensional tensor

Neural network inputs are often represented as rank-2 tensors, even when processing a single example.

For example:

```python
X = np.array([
    [200.0, 12.0]
])

print(X.shape)
# (1, 2)
```

The shape `(1, 2)` means one example with two features.

For a batch of 100 examples with two features each, the shape would be `(100, 2)`.

The first dimension represents the batch size, while the second represents the number of features.

### Converting TensorFlow tensors to NumPy

TensorFlow tensors can be converted to NumPy arrays using `.numpy()` when eager execution is enabled.

```python
tensor = tf.constant([[1.0, 2.0, 3.0]])

array = tensor.numpy()

print(array)
print(array.shape)
```

This is useful when inspecting model outputs, performing NumPy-based analysis, or passing results to code that expects NumPy arrays.

---

## 4. Building Neural Networks with the Sequential API

TensorFlow's `Sequential` API provides a simple way to construct a neural network by arranging layers in order.

Instead of manually calculating each layer's activations, the model automatically passes data from one layer to the next.

```python
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(3, activation="sigmoid"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
```

This architecture accepts two input features, processes them through a hidden layer with three units, and produces one output.

The `Input` layer specifies the shape of each example, excluding the batch dimension.

TensorFlow handles the forward propagation operations internally.

### Why use Sequential?

The Sequential API is useful when the model consists of a straightforward sequence of layers.

Its main advantages include:

- Compact model definitions
- Automatic data flow between layers
- Integration with training and evaluation methods
- Less boilerplate code
- Easier experimentation with layer sizes and activation functions

For models with branching, multiple inputs, or more complex data flows, other Keras APIs may be more appropriate.

---

## 5. Training a Neural Network

Inference uses the model's current parameters to produce predictions. Training adjusts those parameters using examples and their corresponding labels.

A typical supervised learning workflow consists of three stages.

### Step 1: Prepare the data

`X` contains the input features, while `Y` contains the corresponding target labels.

```python
import numpy as np

X = np.array([
    [180.0, 10.0],
    [200.0, 12.0],
    [220.0, 15.0],
    [190.0, 11.0]
], dtype=np.float32)

Y = np.array([
    [0.0],
    [1.0],
    [1.0],
    [0.0]
], dtype=np.float32)
```

These values are illustrative. A real application requires a suitable dataset, reliable labels, and appropriate preprocessing.

### Step 2: Compile the model

The `compile()` method configures how the model will be trained.

```python
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
```

The main components are:

- **Optimizer:** Determines how model parameters are updated.
- **Loss function:** Measures the difference between predictions and target labels.
- **Metrics:** Provide additional measurements of model performance.

Binary cross-entropy is appropriate for a standard binary classification model with a sigmoid output.

### Step 3: Train the model

The `fit()` method trains the model using the provided examples.

```python
model.fit(
    X,
    Y,
    epochs=100,
    verbose=1
)
```

During training, TensorFlow performs forward propagation, calculates the loss, computes gradients through backpropagation, and updates the model parameters.

The number of epochs and other training settings should be selected based on validation performance rather than assumed to be optimal.

---

## 6. Making Predictions with a Trained Model

Once training is complete, `predict()` can generate outputs for new examples.

```python
new_X = np.array([
    [205.0, 13.0]
], dtype=np.float32)

predictions = model.predict(new_X)

print(predictions)
```

For a binary classification model with a sigmoid output, a threshold can be applied to obtain predicted labels:

```python
predicted_labels = (predictions >= 0.5).astype(int)

print(predicted_labels)
```

The prediction workflow is therefore:

```text
Training data (X, Y)
        |
        v
Define model architecture
        |
        v
Compile model
        |
        v
Train with fit()
        |
        v
Trained model
        |
        v
New input data
        |
        v
Predict with predict()
        |
        v
Predictions
```

Training and inference are distinct operations. Training learns the parameters; inference uses those learned parameters to process new inputs.

---

## 7. Understanding the Abstraction

TensorFlow simplifies the implementation of neural networks, but it is still important to understand what happens underneath the API.

A call such as:

```python
prediction = model(X)
```

conceptually performs the following operations:

1. Receives the input tensor.
2. Applies the first layer's weights and biases.
3. Calculates the layer's activations.
4. Passes those activations to the next layer.
5. Repeats the process until the output layer is reached.

TensorFlow manages these operations, including efficient tensor computation and automatic differentiation during training.

Understanding the underlying calculations helps when debugging tensor shapes, interpreting model behavior, and investigating training problems.

Implementing forward propagation manually in Python is a useful complementary exercise because it exposes the calculations that high-level frameworks otherwise hide.

---

## Key Takeaways

- TensorFlow provides layers and APIs for implementing neural network inference and training.
- NumPy arrays and TensorFlow tensors represent numerical data, but their shapes and operations must be understood correctly.
- Neural network inputs commonly use a two-dimensional shape: `(number_of_examples, number_of_features)`.
- `tf.keras.Sequential` provides a straightforward way to connect layers into a model.
- `compile()` configures training, `fit()` learns parameters, and `predict()` generates predictions.
- A model must be trained or supplied with suitable trained parameters before its predictions can be considered meaningful.
- Understanding forward propagation remains valuable even when TensorFlow performs the calculations automatically.

## Next Step

Implement forward propagation manually in Python to understand how the layer calculations work without relying entirely on TensorFlow's abstractions.